## Aggregation Example

In [1]:
# Import the Data generator class from the ts_data_generator module
from ts_data_generator import DataGen
from ts_data_generator.schema.models import AggregationType
from ts_data_generator.utils.functions import random_choice, random_int
from ts_data_generator.utils.trends import (
    SinusoidalTrend,
    LinearTrend,
    WeekendTrend,
    StockTrend,
)
import matplotlib.pyplot as plt
import random

In [2]:


d = DataGen()
d.start_datetime = "2019-01-01"
d.end_datetime = "2019-02-28"
d.to_granularity("h")


d.add_dimension("product", random_choice(["A", "B", "C", "D"]))
d.add_dimension(name="interface", function="X Y Z".split())

d.add_metric(
    name="sinusoidal",
    trends=[
        SinusoidalTrend(name="sine", amplitude=6, freq=3, phase=0, noise_level=1.5)
    ],
    aggregation_type=AggregationType.SUM,
)


d.add_metric(
    name="sinusoidal_linear",
    trends=[
        SinusoidalTrend(name="sine", amplitude=3, freq=5, phase=0, noise_level=1.5),
        LinearTrend(name="linear", slope=30, offset=10, noise_level=1),
    ],
    aggregation_type=AggregationType.SUM,
)


def my_custom_function():
    while True:
        val1 = random.randint(1, 2)
        val2 = random.randint(1, 3)
        # val3 = val1 + val2
        yield (val1, val2)


d.add_multi_items(
    names="val1 val2".split(),
    function=my_custom_function(),
    aggregation_type=[AggregationType.SUM, AggregationType.AVG],
)
d.add_multi_items(
    names="val3 val4".split(),
    function=my_custom_function(),
    aggregation_type=[AggregationType.SUM, AggregationType.AVG],
)





In [3]:
# get data for the first month
first_month_data = d.data[(d.data.index.month == 1)]
first_month_before_agg_val1 = first_month_data[
    (first_month_data["product"] == "A") & (first_month_data["interface"] == "X")
]["val1"].sum()
first_month_before_agg_val2 = first_month_data[
    (first_month_data["product"] == "A") & (first_month_data["interface"] == "X")
]["val2"].mean()


# Now aggregate the data to monthly granularity and check if the aggregated values match the expected values
aggregated_data = d.aggregate(granularity="ME")

first_month_after_agg_val1 = aggregated_data[
    (aggregated_data.index.month == 1)
    & (aggregated_data["product"] == "A")
    & (aggregated_data["interface"] == "X")
]["val1"].sum()
first_month_after_agg_val2 = aggregated_data[
    (aggregated_data.index.month == 1)
    & (aggregated_data["product"] == "A")
    & (aggregated_data["interface"] == "X")
]["val2"].mean()



In [4]:
print(first_month_before_agg_val1 == first_month_after_agg_val1)  # expected to be True
print(first_month_before_agg_val2 == first_month_after_agg_val2)  # expected to be True

True
True
